In [44]:
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [45]:
data={
    'message': [
        "Win a brand new car! Text WIN to 99999 now!",
        "Hi John, are we meeting today?",
        "Exclusive offer just for you. Claim your free prize!",
        "Please call me when you reach office.",
        "Limited-time deal! Buy 1 get 1 free now!",
        "Can you send me the report by evening?",
        "You have been selected for a $1000 gift card!",
        "Lunch at 1 pm?",
        "Congratulations! You’ve won free tickets to Maldives!",
        "Hey, don't forget about the meeting tomorrow.",
        "Lets go for a party this weekend"
    ],
    'label': [
        "Spam", "Valid", "Spam", "Valid", "Spam",
        "Valid", "Spam", "Valid", "Spam", "Valid","Valid"
    ]
}

In [46]:
df=pd.DataFrame(data)
df.head()

,message,label
0,Win a brand new car! Text WIN to 99999 now!,Spam
1,"Hi John, are we meeting today?",Valid
2,Exclusive offer just for you. Claim your free ...,Spam
3,Please call me when you reach office.,Valid
4,Limited-time deal! Buy 1 get 1 free now!,Spam


In [47]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+','', text)
    text = re.sub(r'www\S+','',text)
    text = re.sub(r'a-z\s','',text)
    text = ''.join([word for word in text.split() if word not in stopwords.words('english')])
    return text

In [48]:
df['clean_message']=df['message'].apply(clean_text)
df.head()

,message,label,clean_message
0,Win a brand new car! Text WIN to 99999 now!,Spam,winbrandnewcar!textwin99999now!
1,"Hi John, are we meeting today?",Valid,"hijohn,meetingtoday?"
2,Exclusive offer just for you. Claim your free ...,Spam,exclusiveofferyou.claimfreeprize!
3,Please call me when you reach office.,Valid,pleasecallreachoffice.
4,Limited-time deal! Buy 1 get 1 free now!,Spam,limited-timedeal!buy1get1freenow!


In [49]:
X_train, X_test, y_train, y_test = train_test_split(df['clean_message'],df['label'],test_size=0.3,random_state=42)

In [50]:
vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


In [51]:
model=MultinomialNB()
model.fit(X_train_vec,y_train)

MultinomialNB()

In [52]:
y_pred=model.predict(X_test_vec)
acc=accuracy_score(y_test,y_pred)
cnf_matrix=confusion_matrix(y_test,y_pred)
print("Accuracy:",acc)
print("Confusion Matrix:",cnf_matrix)
classification_report=classification_report(y_test,y_pred)
print("Classification Report:",classification_report)


Accuracy: 0.25
Confusion Matrix: [[1 0]
 [3 0]]
Classification Report:               precision    recall  f1-score   support

        Spam       0.25      1.00      0.40         1
       Valid       0.00      0.00      0.00         3

    accuracy                           0.25         4
   macro avg       0.12      0.50      0.20         4
weighted avg       0.06      0.25      0.10         4



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [60]:
new_msg=[
    'Free Free entry in our draw of RM 3000 prize!! Click Now!! Offer only for this time',
    'Hi John, are we meeting today?',

]
new_clean_msg=[clean_text(msg) for msg in new_msg]
new_vec=vectorizer.transform(new_clean_msg)
predictions=model.predict(new_vec)
print('Predictions are: ')
for msg,prediction in zip(new_msg,predictions):
    print(f"Message: {msg}\nPredicted Label: {prediction}\n")

Predictions are: 
Message: Free Free entry in our draw of RM 3000 prize!! Click Now!! Offer only for this time
Predicted Label: Spam

Message: Hi John, are we meeting today?
Predicted Label: Valid

